# Preprocessing raw dataset

In [1]:
import pandas as pd
import numpy as np
from pyprojroot import here
import utils

In [2]:
processed_data = here('data/processed_data')

# Load data

In [3]:
columns2keep = [
       'government_level', 
       #'institution_abbreviation', 
       #'institution_name',
       'purchasing_unit_name', 
       #'purchasing_unit_representative', 
       'file_code',
       #'file_title', 
       'file_template', 
       'procedure_number', 
       'award_date',
       'advertisement_date', 
       'submission_deadline_date',
       'country_contract_type', 
       'contracting_type', 
       'procedure_type',
       'procedure_venue', 
       'contract_code', 
       #'contract_title',
       'contract_initial_date', 
       'contract_end_date', 
       'contract_price_mx',
       #'currency', 
       #'contract_status', 
       #'modification_agreement',
       #'federal_program_code', 
       'contract_signature_date',
       #'consolidated_purchase', 
       #'multianual_contract',
       #'finance_ministery_code', 
       'supplier_type_alternative',
       'providers_registry_code', 
       'supplier_name', 
       'supplier_type',
       #'supplier_country_code', 
       #'credit_financial_institution',
       'advertisement_url', 
       'df_year', 
       #'file_reference', 
       #'CUCOP_code',
       'legal_fundament', 
       #'contract_control_number', 
       #'contract_description',
       'RFC', 
       #'verified_RFC', 
       #'RFC_original', 
       #'supplier_name_original',
       'purchasing_unit_id'
]

In [4]:
mex_contracts = pd.read_feather(processed_data / 'merged_contracts_raw.feather', columns=columns2keep)

In [5]:
mex_contracts.columns

Index(['government_level', 'purchasing_unit_name', 'file_code',
       'file_template', 'procedure_number', 'award_date', 'advertisement_date',
       'submission_deadline_date', 'country_contract_type', 'contracting_type',
       'procedure_type', 'procedure_venue', 'contract_code',
       'contract_initial_date', 'contract_end_date', 'contract_price_mx',
       'contract_signature_date', 'supplier_type_alternative',
       'providers_registry_code', 'supplier_name', 'supplier_type',
       'advertisement_url', 'df_year', 'legal_fundament', 'RFC',
       'purchasing_unit_id'],
      dtype='object')

In [6]:
mex_contracts.shape

(2315321, 26)

In [7]:
mex_contracts['df_year'].value_counts()

20101112    296408
2017        234221
2016        231282
2015        220710
2021        210124
2019        197001
2014        195638
2018        194191
2022        193944
2013        177991
2020        163811
Name: df_year, dtype: int64

# Recodify

### Procedure type

In [9]:
(mex_contracts['procedure_type'].value_counts(dropna=False)).sort_index()

ADJUDICACIÓN DIRECTA                     605680
Adjudicación Directa Federal            1125632
Adjudicación directa                       1104
CONTRATO ENTRE ENTES PUBLICOS              5582
INVITACIÓN A CUANDO MENOS 3 PERSONAS      50889
Invitación a Cuando Menos 3 Personas     179455
Invitación a cuando menos 3 personas        596
LICITACIÓN PÚBLICA                        86099
Licitación Publica Estatal                  224
Licitación Pública                       241785
Licitación Pública con OSD                  557
OTRAS CONTRATACIONES                      16489
Otro                                        194
PROYECTO DE CONVOCATORIA                    141
Proyecto de Convocatoria                    344
NaN                                         550
Name: procedure_type, dtype: int64

In [10]:
#recodify the procedure type
procedure_type_dict = {
    'Adjudicación Directa Federal' : 'direct_federal',
    'Invitación a Cuando Menos 3 Personas' : 'at_least_three',
    'Licitación Pública' : 'open',
    'Licitación Publica Estatal' : 'open_regional',
    'Licitación Pública con OSD' : 'open_subsecuent_discount',
    'Adjudicación directa': 'direct',
    'Invitación a cuando menos 3 personas': 'at_least_three',
    'Otro': 'other',
    'Proyecto de Convocatoria': 'project_call',
    'ADJUDICACIÓN DIRECTA' : 'direct',
    'LICITACIÓN PÚBLICA': 'open', 
    'INVITACIÓN A CUANDO MENOS 3 PERSONAS' : 'at_least_three',
    'OTRAS CONTRATACIONES' : 'other',
    'CONTRATO ENTRE ENTES PUBLICOS': 'contract_between_public_entities',
    'PROYECTO DE CONVOCATORIA': 'project_call'}

mex_contracts['procedure_type_original'] = mex_contracts['procedure_type']
mex_contracts['procedure_type_standard'] = mex_contracts['procedure_type_original'].map(procedure_type_dict)

procedure_type_dict = {
    'direct' : 'direct',
    'direct_federal' : 'direct',

    'open' : 'open',
    'open_subsecuent_discount' : 'open',
    'open_regional' : 'open',

    'at_least_three': 'at_least_three',

    'contracts_between_public_entities': 'other',
    'other': 'other',
    'project_call': 'open'
}

mex_contracts['procedure_type'] = mex_contracts['procedure_type_standard'].map(procedure_type_dict)

In [11]:
mex_contracts.drop(columns=['procedure_type_standard', 'procedure_type_original'], inplace=True)

In [12]:
mex_contracts['procedure_type'].value_counts(dropna=False)

direct            1732416
open               329150
at_least_three     230940
other               16683
NaN                  6132
Name: procedure_type, dtype: int64

### Contracting type

In [13]:
mex_contracts['country_contract_type'].value_counts(dropna=False)

Nacional                       1826519
Internacional                   206137
Internacional bajo TLC          194550
NaN                              68174
Otro                             19927
Internacional bajo Tratados          7
Internacional Abierto                7
Name: country_contract_type, dtype: int64

In [14]:
#dictionary replacing contracting_type values

contracting_type_dict = {
    'Adquisiciones': 'goods',
    'ADQUISICIONE': 'goods',
    'ADQUISICIONES': 'goods',
    'Servicios': 'services',
    'SERVICIOS': 'services',
    'Obra Pública': 'works',
    'OBRA PÚBLICA': 'works',
    'Servicios Relacionados con la OP': 'services',
    'Servicios relacionados con la OP': 'services',
    'SERVICIOS RELACIONADOS CON LA OBRA' : 'services',
    'Arrendamientos': 'services',
    'ARRENDAMIENTOS': 'services'
}

country_contract_type_dict = {
    'Nacional': 'national',
    'NACIONAL': 'national',
    'Internacional bajo TLC': 'international',
    'Internacional': 'international',
    'Otro':'other',
    'Internacional bajo Tratados': 'international',
    'INTERNACIONAL BAJO LA COBERTURA DE TRATADOS': 'international',
    'Internacional Abierto': 'international',
    'INTERNACIONAL ABIERTO': 'international'}

mex_contracts['supply_type'] = mex_contracts['contracting_type'].replace(contracting_type_dict)
mex_contracts['legal_framework'] = mex_contracts['country_contract_type'].replace(country_contract_type_dict)

In [15]:
mex_contracts['supply_type'].value_counts()

goods       1304189
services     794246
works        209357
Name: supply_type, dtype: int64

In [16]:
mex_contracts['legal_framework'].value_counts()

national         1826519
international     400701
other              19927
Name: legal_framework, dtype: int64

### Supplier size

In [17]:
mex_contracts['supplier_type'].value_counts(dropna=False)

NaN          651274
Micro        525855
Pequeña      490924
Mediana      336944
No MIPYME    288872
              21452
Name: supplier_type, dtype: int64

In [18]:
mex_contracts['supplier_type']= mex_contracts['supplier_type'].replace(r'^\s*$', np.nan, regex=True)

In [19]:
supplier_size_dict = {
    'Micro': 'micro',
    'MICRO': 'micro',
    'Pequeña': 'small',
    'PEQUEÑA': 'small',
    'Mediana': 'medium',
    'MEDIANA': 'medium',    
    'No MIPYME': 'not_mipyme',
    'NO MIPYME': 'not_mipyme',
    'NO ASIGNADO': 'other',
    'GRANDE': 'not_mipyme',
    'COOPERATIVAS': 'other',
}

mex_contracts['supplier_size'] = mex_contracts['supplier_type'].replace(supplier_size_dict)

In [20]:
mex_contracts['supplier_size'].value_counts(dropna=False)

NaN           672726
micro         525855
small         490924
medium        336944
not_mipyme    288872
Name: supplier_size, dtype: int64

### Government level

In [21]:
mex_contracts['government_level'].value_counts()

APF    2095099
GE      155477
GM       64707
GF          38
Name: government_level, dtype: int64

In [22]:
government_level_dict = {
    'APF': 'federal_authority',
    'GE': 'state_authority',
    'GM': 'local_authority',
    'GF': 'federal_authority',
    'EPP': 'local_authority', #Empresa productiva del estado?
    'CI': 'local_authority',
}

mex_contracts['government_level'] = mex_contracts['government_level'].replace(government_level_dict)

In [23]:
mex_contracts['government_level'].value_counts()

federal_authority    2095137
state_authority       155477
local_authority        64707
Name: government_level, dtype: int64

### Supplier name clean

In [24]:
mex_contracts['supplier_name_clean'] = utils.clean_names(mex_contracts, 'supplier_name')

/mnt/sdb1/marti/detecting_corruption_mexico/scripts/dataset_creation/utils.py:85: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will*not* be treated as literal strings when regex=True.
  new_column = new_column.str.replace(r,'')


### purchasing_unit_representative

In [25]:
#mex_contracts['purchasing_unit_representative_clean'] = utils.clean_names(mex_contracts, 'purchasing_unit_representative')

### Legal fundament

In [27]:
legal_fundament_codification = pd.read_excel(here() / 'data' / 'contracts_data' / 'legal_fundament_codification.xlsx')
#clean names
legal_fundament_codification['original'] = utils.clean_names(legal_fundament_codification, 'original')
legal_fundament_codification['standard'] = utils.clean_names(legal_fundament_codification, 'standard')
legal_fundament_codification['standard'] = legal_fundament_codification['standard'].str.strip()
#create dicts
legal_fundament_codification_standard_dict = dict(zip(legal_fundament_codification['original'], legal_fundament_codification['standard']))
legal_fundament_codification_simplified_dict = dict(zip(legal_fundament_codification['original'], legal_fundament_codification['standard_simplified']))
#erase spaces before and after every value in the column 'legal_fundament'
mex_contracts['legal_fundament'] = utils.clean_names(mex_contracts, 'legal_fundament')
#recoding the values
mex_contracts['legal_fundament_standard'] = mex_contracts['legal_fundament'].replace(legal_fundament_codification_standard_dict)
mex_contracts['legal_fundament_simplified'] = mex_contracts['legal_fundament'].replace(legal_fundament_codification_simplified_dict)

### Procedure venue

In [28]:
procedure_venue_dict = {
    'Presencial': 'in-person', 
    'Electrónica': 'electronic',    
    'Mixta': 'mixed',          
    'ELECTRÓNICA': 'electronic',    
    'PRESENCIAL': 'in-person',     
    'MIXTA': 'mixed'          
    }

mex_contracts['procedure_venue'] = mex_contracts['procedure_venue'].replace(procedure_venue_dict)

In [29]:
mex_contracts['procedure_venue'].value_counts() 

in-person     1024228
electronic     772494
mixed          450321
Name: procedure_venue, dtype: int64

### RFC

In [30]:
rfc2023 = pd.read_feather(processed_data / 'contratos2023_RFCs.feather')

In [31]:
rfc2023 = rfc2023.dropna().reset_index(drop=True)
rfc2023 = rfc2023.drop_duplicates().reset_index(drop=True)
rfc2023['supplier_name_clean'] = utils.clean_names(rfc2023, 'supplier_name')

/mnt/sdb1/marti/detecting_corruption_mexico/scripts/dataset_creation/utils.py:85: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will*not* be treated as literal strings when regex=True.
  new_column = new_column.str.replace(r,'')


In [32]:
rfc2023.drop(columns=['supplier_name'], inplace=True)

In [33]:
rfc_contracts = mex_contracts[['RFC', 'supplier_name_clean']].dropna().drop_duplicates().reset_index(drop=True)

In [34]:
allrfc = pd.concat([rfc2023, rfc_contracts], ignore_index=True)

In [35]:
allrfc = allrfc.drop_duplicates().reset_index(drop=True)

In [36]:
rfc_dict = dict(zip(allrfc['supplier_name_clean'], allrfc['RFC']))

In [37]:
mex_contracts['supplier_name_clean'].isnull().sum()

3

In [38]:
mex_contracts['supplier_name_clean'] = np.where(mex_contracts['supplier_name_clean'].isnull(), mex_contracts['RFC'], mex_contracts['supplier_name_clean'])

In [39]:
mex_contracts['supplier_name_clean'].isnull().sum()

0

In [40]:
mex_contracts['RFC'].isnull().sum()

1718561

In [41]:
mex_contracts['RFC'] = mex_contracts['supplier_name_clean'].map(rfc_dict)

In [42]:
mex_contracts['RFC'].isnull().sum()

828402

# Drop duplicates

In [43]:
print(len(mex_contracts['contract_code']) - len(mex_contracts['contract_code'].unique()))

7715


In [44]:
mex_contracts['contract_code'].isnull().sum()

0

In [45]:
mex_contracts[mex_contracts['contract_code'] == ' ']

,government_level,purchasing_unit_name,file_code,file_template,procedure_number,award_date,advertisement_date,submission_deadline_date,country_contract_type,contracting_type,...,df_year,legal_fundament,RFC,purchasing_unit_id,supply_type,legal_framework,supplier_size,supplier_name_clean,legal_fundament_standard,legal_fundament_simplified


In [46]:
print(len(mex_contracts))
print(len(mex_contracts.drop_duplicates()))

2315321
2315321


# Export the dataset

In [47]:
#save the dataframe
mex_contracts.to_feather(processed_data / '1_2_merged_contracts_preprocessed.feather')